In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, f1_score, precision_score, accuracy_score, precision_recall_curve
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch.nn import BCEWithLogitsLoss, Linear, ModuleDict, LeakyReLU
from sklearn.preprocessing import LabelEncoder
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
import time
from scipy.stats import t as t_dist
import scipy.stats as st
import psutil
import os
import networkx as nx
import random
from torch_geometric.utils import degree

# ---------------------------
# Reproducibility
# ---------------------------
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ---------------------------
# Load & clean data
# ---------------------------
nodes = pd.read_csv("nodes.tsv", sep="\t")
edges = pd.read_csv("edges.tsv", sep="\t")

edges["source"] = edges["source"].str.strip()
edges["target"] = edges["target"].str.strip()
edges["metaedge"] = edges["metaedge"].str.strip()
nodes["id"] = nodes["id"].str.strip()

le_node_kind = LabelEncoder()
nodes['kind_encoded'] = le_node_kind.fit_transform(nodes['kind'])
nodes = nodes.drop_duplicates(subset="id").reset_index(drop=True)
assert (nodes['kind_encoded'] >= 0).all(), "Some nodes have invalid kind_encoded values"

le_metaedge = LabelEncoder()
le_metaedge.fit(edges['metaedge'])
edges['metaedge_encoded'] = le_metaedge.transform(edges['metaedge'])

node_id_map = {node_id: idx for idx, node_id in enumerate(nodes["id"])}

full_existing_set = set()
for src, tgt in zip(edges["source"], edges["target"]):
    s = node_id_map.get(src, -1)
    t = node_id_map.get(tgt, -1)
    if s >= 0 and t >= 0:
        full_existing_set.add((s, t))

node_kind_map = dict(zip(nodes['id'], nodes['kind']))
edges['source_kind'] = edges['source'].map(node_kind_map)
edges['target_kind'] = edges['target'].map(node_kind_map)
edges['source_target_kind'] = edges['source_kind'] + '_' + edges['target_kind']

def create_node_features(nodes_df, num_classes):
    num_nodes = len(nodes_df)
    random_features = torch.nn.init.xavier_normal_(torch.rand(num_nodes, 128))
    kind_embeddings = torch.eye(num_classes)[nodes_df['kind_encoded']].float()
    return torch.cat([random_features, kind_embeddings], dim=1)

num_kind_classes = len(le_node_kind.classes_)
node_features = create_node_features(nodes, num_kind_classes)
assert node_features.shape[0] == len(node_id_map), "Dimension mismatch"

tasks = ["CaD", "CrC", "DrD", "DaG"]

try:
    compound_kind = le_node_kind.transform(['Compound'])[0]
    disease_kind = le_node_kind.transform(['Disease'])[0]
    gene_kind = le_node_kind.transform(['Gene'])[0]
except ValueError:
    raise ValueError("Node kinds must include 'Compound', 'Disease', 'Gene'.")

task_to_kinds = {
    "CrC": (compound_kind, compound_kind),
    "DrD": (disease_kind, disease_kind),
    "CaD": (compound_kind, disease_kind),
    "DaG": (disease_kind, gene_kind)
}

kind_to_nodes = {k: [] for k in range(num_kind_classes)}
for idx, kind in enumerate(nodes['kind_encoded']):
    kind_to_nodes[kind].append(idx)

# ---------------------------
# Split data
# ---------------------------
train_edges, temp_edges = train_test_split(
    edges,
    test_size=0.4,
    random_state=seed,
    stratify=edges['source_target_kind']
)
val_edges, test_edges = train_test_split(
    temp_edges,
    test_size=0.5,
    random_state=seed,
    stratify=temp_edges['source_target_kind']
)

train_edges['metaedge_encoded'] = le_metaedge.transform(train_edges['metaedge'])
val_edges['metaedge_encoded'] = le_metaedge.transform(val_edges['metaedge'])
test_edges['metaedge_encoded'] = le_metaedge.transform(test_edges['metaedge'])

def create_data_object(edges_df, node_features, node_id_map, le_metaedge):
    edge_index = torch.tensor([[node_id_map.get(src, -1), node_id_map.get(tgt, -1)]
                              for src, tgt in zip(edges_df["source"], edges_df["target"])], dtype=torch.long).t()
    valid_mask = (edge_index[0] >= 0) & (edge_index[1] >= 0)
    edge_index = edge_index[:, valid_mask]
    edge_attr = torch.tensor(edges_df['metaedge_encoded'].values, dtype=torch.long)[valid_mask]
    return Data(x=node_features, edge_index=edge_index, edge_attr=edge_attr)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_data = create_data_object(train_edges, node_features, node_id_map, le_metaedge).to(device)
val_data = create_data_object(val_edges, node_features, node_id_map, le_metaedge).to(device)
test_data = create_data_object(test_edges, node_features, node_id_map, le_metaedge).to(device)

def split_edges_per_task(edges_df, task, node_id_map):
    task_edges = edges_df[edges_df['metaedge'] == task]
    edge_pairs = [[node_id_map.get(src, -1), node_id_map.get(tgt, -1)] for src, tgt in zip(task_edges["source"], task_edges["target"])]
    valid_mask = [s >= 0 and t >= 0 for s, t in edge_pairs]
    edge_pairs = [pair for i, pair in enumerate(edge_pairs) if valid_mask[i]]
    edge_attrs = torch.tensor(task_edges['metaedge_encoded'].values[valid_mask], dtype=torch.long)
    return torch.tensor(edge_pairs, dtype=torch.long).t(), edge_attrs

task_edges = {}
for task in tasks:
    task_edges[task] = {}
    for split, edges_df in zip(['train', 'val', 'test'], [train_edges, val_edges, test_edges]):
        edge_index, edge_attr = split_edges_per_task(edges_df, task, node_id_map)
        task_edges[task][split] = {'edge_index': edge_index, 'edge_attr': edge_attr}

def ensure_alignment(edge_index, edge_attr):
    if edge_index.size(1) != edge_attr.size(0):
        raise ValueError(f"edge_index and edge_attr must have matching dimensions: {edge_index.size(1)} vs {edge_attr.size(0)}")
    sorted_idx = edge_index[0].argsort()
    edge_index = edge_index[:, sorted_idx]
    edge_attr = edge_attr[sorted_idx]
    assert edge_index.size(1) == edge_attr.size(0), "Mismatch after sorting"
    return edge_index, edge_attr

train_data.edge_index, train_data.edge_attr = ensure_alignment(train_data.edge_index, train_data.edge_attr)
val_data.edge_index, val_data.edge_attr = ensure_alignment(val_data.edge_index, val_data.edge_attr)
test_data.edge_index, test_data.edge_attr = ensure_alignment(test_data.edge_index, test_data.edge_attr)

degrees = degree(train_data.edge_index[0], num_nodes=node_features.shape[0]).numpy()

# ---------------------------
# Utilities
# ---------------------------
def drop_edge(edge_index, edge_attr, drop_rate=0.1):
    num_edges = edge_index.size(1)
    if num_edges == 0:
        return edge_index, edge_attr
    keep_mask = torch.rand(num_edges, device=edge_index.device) > drop_rate
    return edge_index[:, keep_mask], edge_attr[keep_mask]

def sample_hard_negative_edges(model, embeddings, edge_index, num_neg_samples, num_nodes, task, existing_edges=None, candidate_factor=1):
    if existing_edges is None:
        existing_edges = set(zip(edge_index[0].cpu().numpy(), edge_index[1].cpu().numpy()))
    candidate_neg_samples = max(1, num_neg_samples * candidate_factor)
    src_kind, dst_kind = task_to_kinds[task]
    src_nodes = kind_to_nodes[src_kind]
    dst_nodes = kind_to_nodes[dst_kind]
    if not src_nodes or not dst_nodes:
        return torch.tensor([[], []], dtype=torch.long, device=device), torch.tensor([], dtype=torch.long, device=device)

    src_deg = degrees[src_nodes] + 1e-6
    src_probs = src_deg / src_deg.sum()
    dst_deg = degrees[dst_nodes] + 1e-6
    dst_probs = dst_deg / dst_deg.sum()

    candidate_edges = []
    while len(candidate_edges) < candidate_neg_samples:
        src = np.random.choice(src_nodes, candidate_neg_samples, p=src_probs)
        dst = np.random.choice(dst_nodes, candidate_neg_samples, p=dst_probs)
        for s, d in zip(src, dst):
            if (s, d) not in existing_edges and (d, s) not in existing_edges and s != d and len(candidate_edges) < candidate_neg_samples:
                candidate_edges.append([s, d])
    if not candidate_edges:
        return torch.tensor([[], []], dtype=torch.long, device=device), torch.tensor([], dtype=torch.long, device=device)
    candidate_edges = torch.tensor(candidate_edges, dtype=torch.long, device=device).t()
    with torch.no_grad():
        num_candidates = candidate_edges.size(1)
        task_edge_attr = torch.full((num_candidates,), le_metaedge.transform([task])[0], dtype=torch.long, device=device)
        task_edge_type_emb = model.edge_type_embeddings(task_edge_attr)
        scores = model.predict(embeddings, candidate_edges, task_edge_type_emb, task)
    _, top_indices = torch.topk(scores.flatten(), min(num_neg_samples, candidate_edges.size(1)))
    hard_neg_edges = candidate_edges[:, top_indices]
    hard_neg_attr = task_edge_attr[top_indices]
    return hard_neg_edges, hard_neg_attr

def hidden_state_matching_loss_list(student_hiddens, teacher_hiddens, layer_weights=None):
    assert len(student_hiddens) == len(teacher_hiddens)
    if layer_weights is None:
        layer_weights = [1.0] * len(student_hiddens)
    loss = 0.0
    for w, sh, th in zip(layer_weights, student_hiddens, teacher_hiddens):
        loss = loss + w * F.mse_loss(sh, th, reduction='mean')
    return loss

def batch_sampling(edge_pairs, edge_attr, edge_type_emb, batch_size, device):
    num_edges = edge_pairs.size(1)
    if num_edges == 0:
        return edge_pairs, edge_attr, edge_type_emb
    indices = torch.randperm(num_edges)[:min(batch_size, num_edges)]
    return edge_pairs[:, indices].to(device), edge_attr[indices].to(device), edge_type_emb[indices].to(device)

def get_cosine_temperature(epoch, max_epochs, initial_temperature=2.0, min_temperature=1.0):
    progress = epoch / max_epochs
    temperature = min_temperature + (initial_temperature - min_temperature) * (1 + np.cos(np.pi * progress)) / 2
    return temperature

# ---------------------------
# Models
# ---------------------------
class MultiTaskGraphSAGE(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, task_outputs, num_edge_types):
        super(MultiTaskGraphSAGE, self).__init__()
        self.conv1 = SAGEConv(input_dim, hidden_dim, aggr='mean')
        self.conv2 = SAGEConv(hidden_dim, hidden_dim, aggr='mean')
        self.edge_type_embeddings = torch.nn.Embedding(num_edge_types, hidden_dim)
        self.task_heads = ModuleDict({task: torch.nn.Sequential(
            Linear(hidden_dim * 3, hidden_dim),
            LeakyReLU(negative_slope=0.2),
            Linear(hidden_dim, output_dim),
        ) for task, output_dim in task_outputs.items()})
        self.dropout = torch.nn.Dropout(0.2)
        self.leaky_relu = LeakyReLU(negative_slope=0.2)

    def forward(self, x, edge_index, edge_attr, return_hidden=False):
        x = x.clone()
        h1 = self.leaky_relu(self.conv1(x, edge_index))
        h2 = self.leaky_relu(self.conv2(h1, edge_index))
        z = self.dropout(h2)
        edge_bias = self.edge_type_embeddings(edge_attr).mean(dim=0)
        z = z + edge_bias
        if return_hidden:
            return z, [h1, h2]
        return z

    def predict(self, embeddings, edge_pairs, edge_type_emb, task):
        if edge_pairs.size(1) == 0:
            return torch.tensor([], device=embeddings.device)
        assert edge_pairs.size(1) == edge_type_emb.size(0), "Dimension mismatch in edge pairs and embeddings"
        src = embeddings[edge_pairs[0]]
        dst = embeddings[edge_pairs[1]]
        concat = torch.cat([src, dst, edge_type_emb], dim=1)
        logits = self.task_heads[task](concat)
        return logits

class TeacherModel(MultiTaskGraphSAGE):
    def __init__(self, input_dim, hidden_dim, task_outputs, num_edge_types):
        super(TeacherModel, self).__init__(input_dim, hidden_dim, task_outputs, num_edge_types)
        self.log_vars = torch.nn.Parameter(torch.zeros(len(task_outputs)))

class StudentModel(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, task_outputs, num_edge_types, teacher_hidden_dim):
        super(StudentModel, self).__init__()
        self.conv1 = SAGEConv(input_dim, hidden_dim, aggr='mean')
        self.edge_type_embeddings = torch.nn.Embedding(num_edge_types, hidden_dim)
        self.task_heads = ModuleDict({task: torch.nn.Sequential(
            Linear(hidden_dim * 3, hidden_dim),
            Linear(hidden_dim, output_dim),
        ) for task, output_dim in task_outputs.items()})
        self.log_vars = torch.nn.Parameter(torch.zeros(len(task_outputs)))
        self.proj1 = Linear(hidden_dim, teacher_hidden_dim)

    def forward(self, x, edge_index, edge_attr, return_hidden=False):
        x = x.clone()
        h1 = self.conv1(x, edge_index)
        z = h1
        edge_bias = self.edge_type_embeddings(edge_attr).mean(dim=0)
        z = z + edge_bias
        if return_hidden:
            ph1 = self.proj1(h1)
            return z, [ph1]
        return z

    def predict(self, embeddings, edge_pairs, edge_type_emb, task):
        if edge_pairs.size(1) == 0:
            return torch.tensor([], device=embeddings.device)
        assert edge_pairs.size(1) == edge_type_emb.size(0), "Dimension mismatch in edge pairs and embeddings"
        src = embeddings[edge_pairs[0]]
        dst = embeddings[edge_pairs[1]]
        concat = torch.cat([src, dst, edge_type_emb], dim=1)
        logits = self.task_heads[task](concat)
        return logits

# ---------------------------
# Losses
# ---------------------------
class WeightedBCEWithLogitsLoss(torch.nn.Module):
    def __init__(self, pos_weight=None, label_smoothing=0.1):
        super(WeightedBCEWithLogitsLoss, self).__init__()
        self.criterion = BCEWithLogitsLoss(pos_weight=pos_weight)
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        smoothed = targets * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        return self.criterion(logits.view_as(smoothed), smoothed)

def distillation_loss(student_logits, teacher_logits, temperature=2.0):
    s = torch.stack([torch.zeros_like(student_logits), student_logits], dim=-1) / temperature
    t = torch.stack([torch.zeros_like(teacher_logits), teacher_logits], dim=-1) / temperature
    s_logprob = F.log_softmax(s, dim=-1)
    t_prob = F.softmax(t, dim=-1)
    return F.kl_div(s_logprob, t_prob, reduction='batchmean') * (temperature ** 2)

class WeightedLossCombiner(torch.nn.Module):
    def __init__(self, alpha, beta, gamma):
        super(WeightedLossCombiner, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma

    def forward(self, task_loss, distill_loss, hs_loss):
        return self.alpha * task_loss + self.beta * distill_loss + self.gamma * hs_loss

def mean_ci(data, confidence=0.95):
    data = np.array(data)
    mean = np.mean(data)
    sem = st.sem(data)
    h = sem * st.t.ppf((1 + confidence) / 2., len(data)-1)
    return mean, h

def calculate_flops(model, data, edge_index, edge_attr, task_edges, batch_size, num_nodes, is_student=False):
    flops = 0
    input_dim = data.x.shape[1]
    hidden_dim = model.conv1.out_channels
    num_edges = edge_index.shape[1]

    flops_sageconv = num_edges * (input_dim + hidden_dim) * 2
    if is_student:
        flops += flops_sageconv
    else:
        flops += flops_sageconv * 2

    num_edge_types = model.edge_type_embeddings.num_embeddings
    embedding_dim = model.edge_type_embeddings.embedding_dim
    flops_embedding = num_edges * embedding_dim
    flops += flops_embedding

    for task in task_edges:
        task_head = model.task_heads[task]
        flops_linear1 = batch_size * (hidden_dim * 3 * hidden_dim * 2)
        output_dim = 1
        flops_linear2 = batch_size * (hidden_dim * output_dim * 2)
        flops += flops_linear1 + flops_linear2

    flops_activation = num_nodes * hidden_dim * 2
    flops += flops_activation

    if is_student:
        flops_proj1 = num_nodes * hidden_dim * model.proj1.out_features * 2
        flops += flops_proj1

    gflops = flops / 1e9
    return gflops

def get_memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    return mem_info.rss / 1024 / 1024

# ---------------------------
# Training / Eval
# ---------------------------
def train_teacher(model, train_data, task_edges, optimizer, criterion, device, batch_size, iterations=10):
    model.train()
    start_time = time.time()
    total_loss = 0
    num_nodes = train_data.x.shape[0]

    gflops = calculate_flops(model, train_data, train_data.edge_index, train_data.edge_attr, task_edges, batch_size, num_nodes, is_student=False)
    mem_before = get_memory_usage()

    for _ in range(iterations):
        optimizer.zero_grad()
        edge_index, edge_attr = drop_edge(train_data.edge_index, train_data.edge_attr, drop_rate=0.1)
        embeddings = model(train_data.x, edge_index, edge_attr)
        task_losses = {task: torch.zeros(1, device=device, requires_grad=True) for task in tasks}
        for i_task, (task, edges) in enumerate(task_edges.items()):
            pos_pairs = edges['train']['edge_index'].to(device)
            pos_attr = edges['train']['edge_attr'].to(device)
            if pos_pairs.size(1) == 0:
                continue
            pos_type_emb = model.edge_type_embeddings(pos_attr)
            for _start in range(0, pos_pairs.size(1), batch_size):
                b_pos_pairs, b_pos_attr, b_pos_type_emb = batch_sampling(
                    pos_pairs, pos_attr, pos_type_emb, batch_size, device
                )
                b_neg_pairs, b_neg_attr = sample_hard_negative_edges(
                    model, embeddings, train_data.edge_index, b_pos_pairs.size(1), num_nodes, task,
                    existing_edges=full_existing_set
                )
                b_neg_pairs, b_neg_attr = b_neg_pairs.to(device), b_neg_attr.to(device)
                b_neg_type_emb = model.edge_type_embeddings(b_neg_attr)

                pos_logits = model.predict(embeddings, b_pos_pairs, b_pos_type_emb, task)
                neg_logits = model.predict(embeddings, b_neg_pairs, b_neg_type_emb, task)
                all_logits = torch.cat([pos_logits, neg_logits], dim=0)
                all_labels = torch.cat([torch.ones_like(pos_logits), torch.zeros_like(neg_logits)], dim=0)

                loss = criterion(all_logits, all_labels)
                task_losses[task] = task_losses[task] + loss

        task_loss = torch.zeros(1, device=device, requires_grad=True)
        for i, task in enumerate(tasks):
            if task_losses[task].item() > 0:
                weighted = torch.exp(-model.log_vars[i]) * task_losses[task] + 0.5 * model.log_vars[i]
                task_loss = task_loss + weighted

        task_loss.backward()
        total_loss += task_loss.item()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

    mem_after = get_memory_usage()
    return total_loss / iterations, time.time() - start_time, gflops, mem_after - mem_before

def train_student(model, teacher_model, train_data, task_edges, optimizer, criterion, distillation_criterion, device, loss_combiner, batch_size, iterations=10, epoch=0, max_epochs=50, layer_weights=None):
    model.train()
    teacher_model.eval()
    start_time = time.time()
    total_loss = 0
    total_distill_loss = 0
    total_hs_loss = 0
    num_nodes = train_data.x.shape[0]
    temperature = get_cosine_temperature(epoch, max_epochs)

    gflops = calculate_flops(model, train_data, train_data.edge_index, train_data.edge_attr, task_edges, batch_size, num_nodes, is_student=True)
    mem_before = get_memory_usage()

    for _ in range(iterations):
        optimizer.zero_grad()
        edge_index, edge_attr = drop_edge(train_data.edge_index, train_data.edge_attr, drop_rate=0.1)
        with torch.no_grad():
            t_embed, t_hs = teacher_model(train_data.x, edge_index, edge_attr, return_hidden=True)
            t_hs = [t_hs[-1]]
        s_embed, s_hs_proj = model(train_data.x, edge_index, edge_attr, return_hidden=True)

        hs_loss = hidden_state_matching_loss_list(s_hs_proj, t_hs, layer_weights=layer_weights)

        task_losses = {task: torch.zeros(1, device=device, requires_grad=True) for task in tasks}
        distill_losses = {task: torch.zeros(1, device=device, requires_grad=True) for task in tasks}

        for i_task, (task, edges) in enumerate(task_edges.items()):
            pos_pairs = edges['train']['edge_index'].to(device)
            pos_attr = edges['train']['edge_attr'].to(device)
            if pos_pairs.size(1) == 0:
                continue

            for _start in range(0, pos_pairs.size(1), batch_size):
                s_pos_type_emb_full = model.edge_type_embeddings(pos_attr)
                b_pos_pairs, b_pos_attr, b_s_pos_type_emb = batch_sampling(
                    pos_pairs, pos_attr, s_pos_type_emb_full, batch_size, device
                )

                b_neg_pairs, b_neg_attr = sample_hard_negative_edges(
                    model, s_embed, train_data.edge_index, b_pos_pairs.size(1), num_nodes, task,
                    existing_edges=full_existing_set
                )
                b_neg_pairs, b_neg_attr = b_neg_pairs.to(device), b_neg_attr.to(device)
                b_s_neg_type_emb = model.edge_type_embeddings(b_neg_attr)

                b_t_pos_type_emb = teacher_model.edge_type_embeddings(b_pos_attr)
                b_t_neg_type_emb = teacher_model.edge_type_embeddings(b_neg_attr)

                s_pos_logits = model.predict(s_embed, b_pos_pairs, b_s_pos_type_emb, task)
                s_neg_logits = model.predict(s_embed, b_neg_pairs, b_s_neg_type_emb, task)
                s_all_logits = torch.cat([s_pos_logits, s_neg_logits], dim=0)
                all_labels = torch.cat([torch.ones_like(s_pos_logits), torch.zeros_like(s_neg_logits)], dim=0)

                loss_task = criterion(s_all_logits, all_labels)

                with torch.no_grad():
                    t_pos_logits = teacher_model.predict(t_embed, b_pos_pairs, b_t_pos_type_emb, task)
                    t_neg_logits = teacher_model.predict(t_embed, b_neg_pairs, b_t_neg_type_emb, task)
                    t_all_logits = torch.cat([t_pos_logits, t_neg_logits], dim=0)

                loss_distill = distillation_criterion(s_all_logits, t_all_logits, temperature=temperature)

                task_losses[task] = task_losses[task] + loss_task
                distill_losses[task] = distill_losses[task] + loss_distill
                total_distill_loss += loss_distill.item()

        total_hs_loss += hs_loss.item()

        loss_task_sum = torch.zeros(1, device=device, requires_grad=True)
        distill_sum = torch.zeros(1, device=device, requires_grad=True)
        for i, task in enumerate(tasks):
            if task_losses[task].item() > 0:
                weighted_hard = torch.exp(-model.log_vars[i]) * task_losses[task] + 0.5 * model.log_vars[i]
                loss_task_sum = loss_task_sum + weighted_hard
                weighted_distill = torch.exp(-model.log_vars[i]) * distill_losses[task] + 0.5 * model.log_vars[i]
                distill_sum = distill_sum + weighted_distill

        combined_loss = loss_combiner(loss_task_sum, distill_sum, hs_loss)
        combined_loss.backward()
        total_loss += combined_loss.item()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

    mem_after = get_memory_usage()
    return total_loss / iterations, time.time() - start_time, total_hs_loss / iterations, gflops, mem_after - mem_before

def test_model(model, test_data, task_edges, split='test', iterations=10):
    model.eval()
    start = time.time()
    all_metrics = {task: {'auc': [], 'aupr': [], 'accuracy': [], 'f1': [], 'precision': [], 'threshold': []} for task in task_edges}
    num_nodes = test_data.x.shape[0]

    for _ in range(iterations):
        with torch.no_grad():
            emb = model(test_data.x, test_data.edge_index, test_data.edge_attr)
            for task, edges in task_edges.items():
                edge_split = edges[split]['edge_index']
                edge_attr = edges[split]['edge_attr']
                if edge_split.size(1) == 0:
                    for k in all_metrics[task]:
                        all_metrics[task][k].append(0.0 if k != 'threshold' else 0.5)
                    continue
                type_emb_pos = model.edge_type_embeddings(edge_attr)
                pos_logits = model.predict(emb, edge_split, type_emb_pos, task)
                neg_edges, neg_attr = sample_hard_negative_edges(
                    model, emb, test_data.edge_index, edge_split.size(1), num_nodes, task,
                    existing_edges=full_existing_set
                )
                type_emb_neg = model.edge_type_embeddings(neg_attr)
                neg_logits = model.predict(emb, neg_edges, type_emb_neg, task)

                logits = torch.cat([pos_logits, neg_logits]).cpu().numpy()
                labels = np.concatenate([np.ones_like(pos_logits.cpu().numpy()), np.zeros_like(neg_logits.cpu().numpy())])

                precision, recall, thresholds = precision_recall_curve(labels, logits)
                f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
                optimal_threshold = thresholds[np.argmax(f1_scores)] if len(thresholds) > 0 else 0.0
                binary_preds = (logits >= optimal_threshold).astype(int)
                auc = roc_auc_score(labels, logits) if len(np.unique(labels)) > 1 else 0.0
                aupr = average_precision_score(labels, logits) if len(np.unique(labels)) > 1 else 0.0
                acc = accuracy_score(labels, binary_preds)
                f1 = f1_score(labels, binary_preds, zero_division=0)
                prec = precision_score(labels, binary_preds, zero_division=0)

                all_metrics[task]['auc'].append(auc)
                all_metrics[task]['aupr'].append(aupr)
                all_metrics[task]['accuracy'].append(acc)
                all_metrics[task]['f1'].append(f1)
                all_metrics[task]['precision'].append(prec)
                all_metrics[task]['threshold'].append(optimal_threshold)

    final_metrics = {}
    for task in tasks:
        metrics = {}
        for k in ['auc', 'aupr', 'accuracy', 'f1', 'precision']:
            vals = all_metrics[task][k]
            mean, h = mean_ci(vals)
            metrics[k] = {'mean': mean, 'ci': h}
        final_metrics[task] = metrics
    final_metrics['inference_time'] = {'avg': time.time() - start}
    return final_metrics

def validate_model(model, val_data, task_edges, split='val', iterations=10):
    return test_model(model, val_data, task_edges, split, iterations)

def predict_new_links_and_influence(model, data, node_id_map, le_metaedge, tasks, existing_edges, output_file="predicted_links.csv", top_k=10, max_paths=3, max_candidates=5000, approx_samples=100):
    model.eval()
    num_nodes = data.x.shape[0]
    predicted_links = []

    G = nx.Graph()
    edge_index_np = data.edge_index.cpu().numpy()
    G.add_edges_from(zip(edge_index_np[0], edge_index_np[1]))

    G.add_nodes_from(range(num_nodes))

    betweenness = nx.betweenness_centrality(G, k=approx_samples, normalized=True)

    reverse_node_id_map = {v: k for k, v in node_id_map.items()}

    with torch.no_grad():
        embeddings = model(data.x, data.edge_index, data.edge_attr)

        for task in tasks:
            src_kind, dst_kind = task_to_kinds[task]
            src_nodes = kind_to_nodes[src_kind]
            dst_nodes = kind_to_nodes[dst_kind]
            if not src_nodes or not dst_nodes:
                continue
            candidate_edges = [
                [src, dst]
                for src in src_nodes
                for dst in dst_nodes
                if src < dst and (src, dst) not in existing_edges and (dst, src) not in existing_edges
                and src in G.nodes and dst in G.nodes]

            if not candidate_edges:
                continue

            if len(candidate_edges) > max_candidates:
                candidate_edges = random.sample(candidate_edges, max_candidates)

            candidate_edges = torch.tensor(candidate_edges, dtype=torch.long, device=device).t()
            task_edge_attr = torch.full((candidate_edges.size(1),), le_metaedge.transform([task])[0],
                dtype=torch.long, device=device)
            task_edge_type_emb = model.edge_type_embeddings(task_edge_attr)

            scores = torch.sigmoid(model.predict(embeddings, candidate_edges, task_edge_type_emb, task)).flatten()

            top_scores, top_indices = torch.topk(scores, k=min(top_k, scores.size(0)))
            new_edges = candidate_edges[:, top_indices].cpu().numpy().T
            new_scores = top_scores.cpu().numpy()

            for i, (src, dst) in enumerate(new_edges):
                src_name, dst_name = reverse_node_id_map[src], reverse_node_id_map[dst]

                try:
                    paths = list(nx.all_shortest_paths(G, src, dst))
                    paths = paths[:max_paths]
                    intermediate_nodes = {n for p in paths for n in p[1:-1]}
                    for node in intermediate_nodes:
                        predicted_links.append({
                            'task': task,
                            'source': src_name,
                            'target': dst_name,
                            'score': new_scores[i],
                            'intermediate_node': reverse_node_id_map[node],
                            'node_influence': betweenness.get(node, 0.0),
                            'num_paths': len(paths)
                        })
                except nx.NetworkXNoPath:
                    predicted_links.append({
                        'task': task,
                        'source': src_name,
                        'target': dst_name,
                        'score': new_scores[i],
                        'intermediate_node': None,
                        'node_influence': None,
                        'num_paths': 0
                    })

    if predicted_links:
        predicted_df = pd.DataFrame(predicted_links)
        predicted_df.to_csv(output_file, index=False)
        print(f"Predicted links saved to {output_file} (total {len(predicted_links)})")
    else:
        print("No new links predicted.")

    return predicted_links

def print_metrics(prefix, metrics):
    for task in tasks:
        if task in metrics:
            print(f"{prefix} {task}: ", end='')
            for k, v in metrics[task].items():
                print(f"{k.upper()}: {v['mean']:.4f} ± {v['ci']:.4f} ", end='')
            print()

# ---------------------------
# Hyperparams & Train
# ---------------------------
hidden_dim_t = 128
hidden_dim_s = 64
task_outputs = {task: 1 for task in tasks}
num_edge_types = len(le_metaedge.classes_)
batch_size = 256
epochs_t = 100
epochs_s = 50
manual_pos_weight = 1.0

teacher_model = TeacherModel(input_dim=node_features.shape[1], hidden_dim=hidden_dim_t, task_outputs=task_outputs, num_edge_types=num_edge_types).to(device)
optimizer_teacher = torch.optim.Adam(teacher_model.parameters(), lr=0.0008, weight_decay=0.0005)
criterion = WeightedBCEWithLogitsLoss(label_smoothing=0.1)
criterion.criterion.pos_weight = torch.tensor([manual_pos_weight], device=device)

teacher_gflops_list = []
teacher_mem_list = []

print("Training Teacher Model...")
for epoch in range(epochs_t):
    loss_t, t_time, gflops_t, mem_usage_t = train_teacher(teacher_model, train_data, task_edges, optimizer_teacher, criterion, device, batch_size)
    teacher_gflops_list.append(gflops_t)
    teacher_mem_list.append(mem_usage_t)
    val_metrics = validate_model(teacher_model, val_data, task_edges)
    print(f"Epoch {epoch + 1}/{epochs_t} - Loss: {loss_t:.4f}, Time: {t_time:.4f}s, GFLOPs: {gflops_t:.4f}, Memory Usage (MB): {mem_usage_t:.2f}")
    print_metrics("Val Teacher", val_metrics)

alpha, beta, gamma = 0.5, 0.3, 0.2
student_model = StudentModel(input_dim=node_features.shape[1], hidden_dim=hidden_dim_s, task_outputs=task_outputs, num_edge_types=num_edge_types, teacher_hidden_dim=hidden_dim_t).to(device)
optimizer_student = torch.optim.Adam(student_model.parameters(), lr=0.0008, weight_decay=0.0005)
loss_combiner = WeightedLossCombiner(alpha=alpha, beta=beta, gamma=gamma).to(device)

student_gflops_list = []
student_mem_list = []

print(f"\nTraining Student Model (alpha={alpha:.2f}, beta={beta:.2f}, gamma={gamma:.2f})...")
for epoch in range(epochs_s):
    loss_s, s_time, hs_l, gflops_s, mem_usage_s = train_student(student_model, teacher_model, train_data, task_edges, optimizer_student, criterion, distillation_loss, device, loss_combiner, batch_size, epoch=epoch, max_epochs=epochs_s, layer_weights=[1.0])
    student_gflops_list.append(gflops_s)
    student_mem_list.append(mem_usage_s)
    val_metrics = validate_model(student_model, val_data, task_edges)
    print(f"Epoch {epoch + 1}/{epochs_s} - Loss: {loss_s:.4f}, Time: {s_time:.4f}s, HS Loss: {hs_l:.4f}, GFLOPs: {gflops_s:.4f}, Memory Usage (MB): {mem_usage_s:.2f}")
    print_metrics("Val Student", val_metrics)

test_metrics_student = test_model(student_model, test_data, task_edges, 'test')
test_metrics_teacher = test_model(teacher_model, test_data, task_edges, 'test')

print("\nStudent Model Test Results:")
print_metrics("Test Student", test_metrics_student)
print(f"Inference Time: {test_metrics_student['inference_time']['avg']:.4f}s")

print("\nTeacher Model Test Results:")
print_metrics("Test Teacher", test_metrics_teacher)
print(f"Inference Time: {test_metrics_teacher['inference_time']['avg']:.4f}s")

avg_teacher_gflops = np.mean(teacher_gflops_list)
avg_teacher_mem = np.mean(teacher_mem_list)
avg_student_gflops = np.mean(student_gflops_list)
avg_student_mem = np.mean(student_mem_list)

print("\nComparative Averages:")
print(f"Average GFLOPs - Teacher: {avg_teacher_gflops:.4f}, Student: {avg_student_gflops:.4f}")
print(f"Average Memory Usage (MB) - Teacher: {avg_teacher_mem:.2f}, Student: {avg_student_mem:.2f}")

print("\nPredicting new links and calculating node influence...")
predict_new_links_and_influence(student_model, test_data, node_id_map, le_metaedge, tasks, full_existing_set, output_file="predicted_links.csv")